# Support Vector Machine

Notebook này tái lập kết quả **train/CV trên tập C** bằng pipeline đã khóa cho mô hình Support Vector Machine. Siêu tham số được chọn bằng CV trên train; hold-out chưa được nạp hoặc chấm trong notebook này.

**Mục tiêu:** ghi lại quy trình, chỉ số và nhận xét cho riêng mô hình. CV này là đánh giá sau lựa chọn siêu tham số, không phải điểm test độc lập.


## 1 Thiết lập

- Phân chia train/hold-out: phân tầng, 80/20, seed 42; chỉ lấy phần train.
- Đánh giá: Stratified 5-fold CV, shuffle=True, seed 42.
- Độ đo chính: Macro-F1; kèm balanced accuracy, accuracy, recall Dropout và F1 Enrolled.
- Quy tắc đầu ra: `native_predict`; không áp ngưỡng tùy chỉnh trong bảng so sánh chính.


In [1]:
from pathlib import Path
import json
import warnings

import joblib
import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "src" / "config.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run this notebook inside the project checkout.")
import os, sys
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src import config
from src.data import get_train_test
from src.evaluate import cross_validate_model
from src.predict import predict_one

MODEL_KEY = "svm"
MODEL_NAME = "Support Vector Machine"
MODEL_PATH = config.MODELS_DIR / "svm_best_proba.joblib"
assert MODEL_PATH.is_file(), MODEL_PATH
X_train, _, y_train, _ = get_train_test("C")  # bỏ qua hold-out
model = joblib.load(MODEL_PATH)
assert list(model.classes_) == config.CLASSES
print(f"Model: {MODEL_NAME} | feature set C | train rows: {len(X_train)} | artifact: {MODEL_PATH.name}")


Model: Support Vector Machine | feature set C | train rows: 3539 | artifact: svm_best_proba.joblib


## 2 Kết quả CV trên train

Ô tiếp theo fit clone của pipeline trên từng fold. Tập hold-out không được sử dụng.

In [2]:
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="The `probability` parameter")
    result = cross_validate_model(model, X_train, y_train, name=MODEL_NAME,
                                  feature_set="C", log=False, n_jobs=1)
metrics = pd.DataFrame([result])
display(metrics[["f1_macro_mean", "f1_macro_std", "balanced_accuracy_mean",
                 "accuracy_mean", "recall_dropout_mean", "f1_enrolled_mean",
                 "train_f1_macro_mean"]].round(4))


,f1_macro_mean,f1_macro_std,balanced_accuracy_mean,accuracy_mean,recall_dropout_mean,f1_enrolled_mean,train_f1_macro_mean
0,0.7213,0.0113,0.7294,0.7601,0.7273,0.5398,0.7691


## 3 Ví dụ dự đoán và nhận xét

Ví dụ lấy từ một dòng train để kiểm tra hợp đồng nhãn/xác suất. Đây không phải mẫu test.

In [3]:
example = X_train.iloc[0].to_dict()
prediction = predict_one(example, model=MODEL_KEY, feature_set="C")
display(prediction)

macro = float(metrics.loc[0, "f1_macro_mean"])
macro_sd = float(metrics.loc[0, "f1_macro_std"])
recall_dropout = float(metrics.loc[0, "recall_dropout_mean"])
f1_enrolled = float(metrics.loc[0, "f1_enrolled_mean"])
gap = float(metrics.loc[0, "train_f1_macro_mean"] - macro)
print("Nhận xét:")
print(f"- Macro-F1 CV = {macro:.4f} ± {macro_sd:.4f}; Recall Dropout = {recall_dropout:.4f}; F1 Enrolled = {f1_enrolled:.4f}.")
if gap > 0.10:
    print(f"- Chênh lệch train–CV = {gap:.4f}; có dấu hiệu overfit cần nêu khi so sánh.")
else:
    print(f"- Chênh lệch train–CV = {gap:.4f}; vẫn đối chiếu với biến động giữa các fold.")
print("- CV sau lựa chọn cấu hình trên train có thể lạc quan; không thay thế hold-out.")


{'label': 'Graduate',
 'probabilities': {'Dropout': 0.06248783391492473,
  'Enrolled': 0.08701174155512874,
  'Graduate': 0.8505004245299467},
 'model': 'svm',
 'feature_set': 'C',
 'decision_rule': 'native_predict',
 'threshold': None,
 'warnings': []}

Nhận xét:
- Macro-F1 CV = 0.7213 ± 0.0113; Recall Dropout = 0.7273; F1 Enrolled = 0.5398.
- Chênh lệch train–CV = 0.0478; vẫn đối chiếu với biến động giữa các fold.
- CV sau lựa chọn cấu hình trên train có thể lạc quan; không thay thế hold-out.


## 4 Nguồn và phạm vi

Giao thức và kết quả A/B/C nằm trong `notebooks/Experiments/Logs/Experiment_04_Timepoint_ABC/timepoint_cv.csv`. Kết quả train/CV cho cấu hình C được lưu ở bảng metrics riêng của từng model; GB/SVM có thêm `reports/sv4_rebuild_cv.csv`. Không điều chỉnh mô hình/ngưỡng theo hold-out. Kết quả cuối chỉ được thêm sau khi manifest được khóa và preflight đạt.
